# 07-early-stop — Luật dừng sớm theo det Σ

**Câu hỏi**: luật dừng sớm trong code so sánh cái gì, ngưỡng của lượt F và lượt S là bao nhiêu, khác Eq. 11 của paper thế nào, và ngưỡng có giải thích được bước dừng của lượt S không? (D14, Q07.02, QD24)

### Nguồn gốc (provenance)

In commit và trạng thái dirty của code, rồi đặt `XLA_FLAGS` trước mọi lần import jax (mẫu ở `lab/README.vi.md` §11).

In [1]:
# cell 0: provenance (run first)
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])

commit daf9589 | code dirty False | XLA_FLAGS --xla_gpu_deterministic_ops=true


- **Kết quả**: commit `daf9589`, code dirty False, XLA_FLAGS `--xla_gpu_deterministic_ops=true`.

## 1. Kiểm

### 1.1 Chạy lại bước 07

`s07_early_stop` không tính gì. Nó chỉ trả về tên của luật dừng (`stop_default`). Bước 11 và bước 13 gọi luật này trước mỗi lần chạy SVI, và luật chạy ở mỗi bước bên trong `run_inference`. Dùng `replay.rerun` để kiểm khoá `stop` trùng với ảnh chụp, và in trạng thái mà `stop_default` đặt lên bộ phân lớp (`with_early_stop`).

In [2]:
# cell 1: step 07 again from the snapshot of step 06 (replay.rerun, QD23): it only names the early-stop rule used by 11 and 13
RUN = "gc10_v2_ft_r0_s4"
out07, ok07 = replay.rerun(RUN, "07")
assert ok07, "step 07 does not reproduce the snapshot"
args_11 = replay.snapshots(RUN)["04"]["meta"]["args"]
print("stop component:", out07["stop"], "| with_early_stop", args_11["with_early_stop"],
      "| use_correct_scaling_factor", args_11["use_correct_scaling_factor"], "| scaling_factor", args_11["scaling_factor"],
      "| early_stop_ratio", args_11["early_stop_ratio"], "| n_epochs", args_11["n_epochs"])

/home/rito/miniforge3/envs/vbcgcd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rerun step 07 of gc10_v2_ft_r0_s4 | device gpu | GPU now NVIDIA GeForce RTX 4060 Laptop GPU, 621.01 | GPU of the run NVIDIA GeForce RTX 4060 Laptop GPU, 617.42 | driver differs: float outputs may move in the last bits
  OK   stop         str                    max|diff| nan
rerun step 07: OK (1/1 keys)
stop component: stop_default | with_early_stop True | use_correct_scaling_factor True | scaling_factor 1.2 | early_stop_ratio 0 | n_epochs 1000


- **Kết quả**: `rerun` bước 07 đạt 1/1 khoá: `stop` = `stop_default`. Các tham số của run: `with_early_stop` True, `use_correct_scaling_factor` True, `scaling_factor` 1.2, `early_stop_ratio` 0, `n_epochs` 1000.
- **Ý nghĩa**: bước 07 chỉ chọn luật dừng. Việc tính toán diễn ra ở 11 và 13. Có hai tham số trông như liên quan tới luật dừng là `scaling_factor` (1.2) và `early_stop_ratio` (0); §2.1 kiểm xem code có đọc chúng không.

## 2. Code

### 2.1 Luật dừng trong code, và hai cờ có được đọc không

In nguyên văn bằng `inspect.getsource`:
- `replay.stop_default`: bật hoặc tắt luật theo `with_early_stop`;
- `MNGMMClassifier.calculate_metrics_on_covariances`: tính hai giá trị det và cờ dừng;
- `MNGMMClassifier._correct_scaling_factors`;
- các dòng của `run_inference` quyết định việc dừng.

Sau đó quét mã nguồn của `classifier/mngmm.py` và `main.py`, tìm mọi chỗ *đọc* `self.scaling_factor`, `args.scaling_factor`, `self.early_stop_ratio`, `args.early_stop_ratio`. Chỉ tính chỗ đọc; phép gán `self.x = ...` không tính.

In [3]:
# cell 2: the early-stop code as it is (inspect.getsource), the stop lines of run_inference, and every place that READS
#         scaling_factor or early_stop_ratio in classifier/mngmm.py and main.py (Q07.02)
import inspect, re
from classifier.mngmm import MNGMMClassifier
for f_22 in (replay.stop_default, MNGMMClassifier.calculate_metrics_on_covariances, MNGMMClassifier._correct_scaling_factors):
    print(f"# ---- {Path(inspect.getsourcefile(f_22)).name}: {f_22.__qualname__}\n{inspect.getsource(f_22)}")
print("# ---- run_inference, stop lines:")
for line_22 in inspect.getsource(MNGMMClassifier.run_inference).splitlines():
    if re.search(r"early_stop|last_state =|self.svi_state = last_state|break", line_22):
        print(line_22.rstrip())
for file_22 in ("classifier/mngmm.py", "main.py"):
    for n_22, line_22 in enumerate(Path(file_22).read_text().splitlines(), 1):
        for name_22 in ("scaling_factor", "early_stop_ratio"):
            for m_22 in re.finditer(rf"(self|args)\.{name_22}\b", line_22):
                assigned = re.match(rf"\s*(self|args)\.{name_22}\s*=[^=]", line_22[m_22.start():]) is not None
                print(f"{file_22}:{n_22} {'assign' if assigned else 'READ  '} {line_22.strip()[:110]}")

# ---- replay.py: stop_default
def stop_default(clf, args):
    """Early-stop rule: mngmm's det(Sigma) rule, on when --with_early_stop (inactive at S0)."""
    clf.with_early_stop = args.with_early_stop
    clf.__dict__.pop("calculate_metrics_on_covariances", None)       # drop an override left by a variant

# ---- mngmm.py: MNGMMClassifier.calculate_metrics_on_covariances
    def calculate_metrics_on_covariances(self, params, increment, use_correct_scaling_factor):
        class_covs = params["class_covs"]

        early_stop_flag = False

        if self.global_params is None:
            return early_stop_flag, [jnp.ones(1),
                                     jnp.ones(1)]

        global_class_covs = self.global_params["class_covs"]

        if not use_correct_scaling_factor:
            dets = [(jax.vmap(jnp.linalg.det)(global_class_covs[:self.num_base])).mean(), 
                (jax.vmap(jnp.linalg.det)(class_covs[self.label_offset: self.label_offset + increment])).mean()]
    

- **Kết quả**:
  - `stop_default` chỉ đặt `with_early_stop` và gỡ phép thay hàm mà một biến thể có thể để lại.
  - `calculate_metrics_on_covariances`:
    - nếu `global_params` là None (phiên S0) thì trả về False, cùng hai số det đều bằng 1;
    - nếu không, `dets[0]` là det trung bình của các lớp cũ, lấy từ `global_params`, và `dets[1]` là det trung bình của các khe lớp mới đang học;
    - khi `use_correct_scaling_factor` False (lượt F), lớp cũ là `[:num_base]` và hệ số sf = (label_offset + increment)/num_base;
    - khi True (lượt S), lớp cũ là `[:label_offset]` và sf = √(increment/(label_offset + increment)), tính bằng `_correct_scaling_factors`;
    - cờ dừng bật khi (det cũ > 1 và det mới > sf × det cũ), hoặc khi (det cũ < 1 và det mới < det cũ / sf).
  - Trong `run_inference`, cờ được kiểm ở đầu mỗi bước, trên tham số sau t lần cập nhật. Luật chỉ dừng khi `step > 1/3 · num_steps`, tức từ bước 334. Khi dừng, trạng thái quay về `last_state`, tức tham số sau t − 1 lần cập nhật.
  - `scaling_factor` và `early_stop_ratio`: trong `mngmm.py`, cả hai chỉ được gán (dòng 64, 68). Trong `main.py`, `args.scaling_factor` chỉ được in ra (dòng 170) và truyền vào `init_parameters` (dòng 176). Không chỗ nào khác đọc hai giá trị này.
- **Ý nghĩa**:
  - Trả lời Q07.02: `--scaling-factor 1.2` không tác động vào phép tính nào. Hệ số của luật dừng tính tại chỗ từ số lớp, với GC10 là 2 ở lượt F và √(5/10) ≈ 0.707 ở lượt S. Như vậy `--scaling-factor` là một cờ chết, giống `--early_stop_ratio` (D14).
  - Luật so det *trung bình* của 5 lớp mới với det trung bình của lớp cũ. Paper (Eq. 11) xét từng lớp mới k. §2.2 so hai cách bằng số.
  - Vì có điều kiện bước > 333, mọi lần dừng đều xảy ra sau ít nhất 1/3 ngân sách bước.

### 2.2 Ngưỡng của lượt F và lượt S, so với Eq. 11 của paper

Tính từ `params0` của r0 s4. Các khe lớp mới bắt đầu ở Σ = I, tức det = 1, và det cũ là 0.3707 < 1 (06 §3.2). Vậy nhánh đang chạy là "det cũ < 1": luật dừng khi det trung bình của lớp mới giảm xuống dưới det cũ / sf.

**Cách làm**:
- Code (D14):
  - lượt F: sf = (5 + 5)/5 = 2, nên ngưỡng = det cũ / 2;
  - lượt S: sf = √(5/10) ≈ 0.7071, nên ngưỡng = det cũ / 0.7071 ≈ 1.414 × det cũ.
- Paper, Eq. 11: R = log((k − 1) det Σ_k / Σ_{i<k} det Σ_i), dừng khi R < ε = 0.01. Ở đây Σ_{i<k} det Σ_i / (k − 1) là det trung bình của các lớp cũ, nên R < 0.01 tương đương det Σ_k < e^0.01 × det cũ ≈ 1.010 × det cũ.

Ví dụ với det cũ = 0.3707: ngưỡng F = 0.3707/2 = 0.1854; ngưỡng S = 0.3707/0.7071 = 0.5243; ngưỡng Eq. 11 = 1.010 × 0.3707 = 0.3744. Khi det lớp mới đi xuống từ 1, lượt S chạm ngưỡng trước, rồi tới Eq. 11, cuối cùng mới tới lượt F.

Cell tính lại các số này từ `snap/06.pkl`, dùng det float32 của JAX như code.

In [4]:
# cell 3: stop thresholds of pass F and pass S from params0 of the snapshot (code, D14), against Eq. 11 of the paper (R < 0.01)
import numpy as np, jax, jax.numpy as jnp
p0_33 = replay.snapshots(RUN)["06"]["out"]["params0"]
covs_33 = jnp.asarray(p0_33["class_covs"])
base_33, inc_33, off_33 = args_11["base"], args_11["increment"], args_11["base"]          # 5, 5, label_offset at S1 = 5
det_old_F = float(jax.vmap(jnp.linalg.det)(covs_33[:base_33]).mean())                     # use_correct_scaling_factor False
det_old_S = float(jax.vmap(jnp.linalg.det)(covs_33[:off_33]).mean())                      # True
sf_F, sf_S = (off_33 + inc_33) / base_33, float(np.sqrt(inc_33 / (off_33 + inc_33)))
det_new0 = float(jax.vmap(jnp.linalg.det)(covs_33[off_33:off_33 + inc_33]).mean())        # new slots at S1 start: Sigma = I
print(f"det old (F) {det_old_F:.4f} | det old (S) {det_old_S:.4f} | det of the new slots at the start {det_new0:.4f}")
for name_33, d_old, sf in (("pass F", det_old_F, sf_F), ("pass S", det_old_S, sf_S)):
    branch = "det old < 1: stop when det new < det old / sf" if d_old < 1 else "det old > 1: stop when det new > sf * det old"
    thr = d_old / sf if d_old < 1 else sf * d_old
    print(f"{name_33}: sf {sf:.4f} | {branch} | threshold {thr:.4f} ({thr / d_old:.3f} x det old)")
print(f"Eq. 11 (R < 0.01): stop when det new < {np.exp(0.01) * det_old_S:.4f} ({np.exp(0.01):.3f} x det old)")

det old (F) 0.3707 | det old (S) 0.3707 | det of the new slots at the start 1.0000
pass F: sf 2.0000 | det old < 1: stop when det new < det old / sf | threshold 0.1853 (0.500 x det old)
pass S: sf 0.7071 | det old < 1: stop when det new < det old / sf | threshold 0.5242 (1.414 x det old)
Eq. 11 (R < 0.01): stop when det new < 0.3744 (1.010 x det old)


- **Kết quả**:
  - det trung bình của lớp cũ là 0.3707 ở cả hai lượt. Với GC10 t1, `[:num_base]` và `[:label_offset]` là cùng 5 lớp. Det trung bình của các khe lớp mới lúc bắt đầu là 1.0000.
  - Lượt F: sf = 2, nhánh "det cũ < 1", ngưỡng 0.1853 (0.5 × det cũ).
  - Lượt S: sf = 0.7071, cùng nhánh, ngưỡng 0.5242 (1.414 × det cũ).
  - Eq. 11 (R < 0.01): dừng khi det lớp mới < 0.3744 (1.010 × det cũ).
- **Ý nghĩa**:
  - Số khớp với phép tính tay và với notebook cũ (0.5242). Ở GC10, hai lượt chỉ khác nhau ở sf, không khác ở det cũ.
  - Khi det trung bình của lớp mới giảm từ 1 xuống, ba ngưỡng xếp theo thứ tự: S (0.524) > Eq. 11 (0.374) > F (0.185). Lượt S dừng sớm hơn Eq. 11: nó dừng khi det lớp mới còn lớn hơn det cũ 41%, trong khi paper dừng khi hai det xấp xỉ bằng nhau. Lượt F dừng muộn nhất, khi det lớp mới chỉ còn một nửa det cũ.
  - Paper lập luận rằng dừng khi hiệp phương sai của lớp mới bằng hiệp phương sai của lớp cũ thì cân bằng được ổn định và thích nghi (Eq. 9–11). Code dừng lượt S ở một điểm khác, nên lập luận đó không mô tả đúng việc code làm (D14).
  - [SUY LUẬN] Vì det cũ phần lớn do ngân sách tối ưu quyết định (06 §3.2), và det lớp mới cũng giảm từ 1 với tốc độ do ngân sách bước chặn, bước dừng của lượt S được quyết định chủ yếu bởi lịch learning rate cộng với hệ số 1.414. 13 sẽ kiểm bằng quỹ đạo det.

## 3. Qua 15 ảnh chụp

### 3.1 Ngưỡng của lượt S so với bước dừng ghi trong metrics.json

Với mỗi ảnh chụp, tính det cũ từ `snap/06.pkl`, rồi tính ngưỡng S = det cũ / 0.7071 và ngưỡng F = det cũ / 2. Đọc bước dừng từ `metrics.json` của chính run đó: `S1_estop_S` là bước dừng của lượt S; khoá `S1_estop_F` chỉ có khi lượt F dừng.

**Cách làm**: tương quan hạng Spearman giữa ngưỡng S và bước dừng S, tính trên cả 15 ảnh chụp và trong từng FT seed (n = 5, chỉ để mô tả). Dự đoán [SUY LUẬN, từ 06 §4.1]: ngưỡng càng cao thì dừng càng sớm, tức tương quan âm, và phần lớn tương quan đến từ khác biệt giữa các FT seed. Bước dừng đã biết từ trước (06 §4.1 có nhắc), nên đây là phép kiểm nhất quán, không phải dự đoán mù.

In [5]:
# cell 4: across the 15 snapshots: old-class det (snap/06.pkl) and the F / S thresholds, against the stop steps in metrics.json
import pickle, json
import pandas as pd
from scipy.stats import spearmanr
rows_44 = []
for d in sorted(Path("runs").glob("*__gc10_v2_ft_r[0-2]_s[0-4]")):
    covs = np.asarray(pickle.loads((d / "snap" / "06.pkl").read_bytes())["out"]["params0"]["class_covs"], np.float64)
    det_old = float(np.mean([np.linalg.det(covs[c]) for c in range(5)]))
    m = json.loads((d / "metrics.json").read_text())
    rows_44.append({"run": d.name.split("__")[1][len("gc10_v2_ft_"):], "det old": round(det_old, 4),
                    "threshold S": round(det_old / sf_S, 4), "threshold F": round(det_old / sf_F, 4),
                    "stop S": m.get("S1_estop_S"), "stop F": m.get("S1_estop_F", "none")})
df_44 = pd.DataFrame(rows_44)
print(df_44.to_string(index=False))
rho = spearmanr(df_44["threshold S"], df_44["stop S"])
print(f"Spearman(threshold S, stop S), all 15: {rho.statistic:.3f} (p = {rho.pvalue:.3g})")
for r in range(3):
    sub = df_44[df_44["run"].str.startswith(f"r{r}_")]
    print(f"  r{r}: threshold S {sub['threshold S'].min()}-{sub['threshold S'].max()} | stop S {sub['stop S'].min()}-{sub['stop S'].max()}"
          f" | Spearman {spearmanr(sub['threshold S'], sub['stop S']).statistic:.2f}")

  run  det old  threshold S  threshold F  stop S stop F
r0_s0   0.3708       0.5244       0.1854     619   none
r0_s1   0.3704       0.5238       0.1852     620   none
r0_s2   0.3723       0.5265       0.1861     593   none
r0_s3   0.3719       0.5260       0.1860     531   none
r0_s4   0.3707       0.5242       0.1853     526   none
r1_s0   0.4192       0.5929       0.2096     543   none
r1_s1   0.4174       0.5903       0.2087     549   none
r1_s2   0.4206       0.5949       0.2103     534   none
r1_s3   0.4188       0.5923       0.2094     486   none
r1_s4   0.4194       0.5932       0.2097     468   none
r2_s0   0.4254       0.6016       0.2127     503   none
r2_s1   0.4235       0.5990       0.2118     507   none
r2_s2   0.4264       0.6031       0.2132     493   none
r2_s3   0.4246       0.6004       0.2123     475   none
r2_s4   0.4251       0.6012       0.2126     490   none
Spearman(threshold S, stop S), all 15: -0.657 (p = 0.00777)
  r0: threshold S 0.5238-0.5265 | stop S 526

- **Kết quả**:

  | FT seed | det cũ | ngưỡng S | ngưỡng F | bước dừng S, seed 0–4 | lượt F dừng |
  |---|---|---|---|---|---|
  | r0 | 0.3704–0.3723 | 0.5238–0.5265 | 0.1852–0.1861 | 619, 620, 593, 531, 526 | không (5/5) |
  | r1 | 0.4174–0.4206 | 0.5903–0.5949 | 0.2087–0.2103 | 543, 549, 534, 486, 468 | không (5/5) |
  | r2 | 0.4235–0.4264 | 0.5990–0.6031 | 0.2118–0.2132 | 503, 507, 493, 475, 490 | không (5/5) |

  Tương quan Spearman giữa ngưỡng S và bước dừng S:
  - trên cả 15 ảnh chụp: −0.657 (p = 0.0078);
  - trong từng FT seed: r0 −0.30, r1 −0.50, r2 −0.10.
- **Ý nghĩa**:
  - Lượt F không dừng ở ảnh chụp nào (15/15), khớp với ngưỡng F thấp (khoảng 0.19–0.21). Lượt S dừng ở mọi ảnh chụp, từ bước 468 tới 620.
  - Đúng như dự đoán, ngưỡng càng cao thì dừng càng sớm. Nhưng tương quan −0.66 gần như hoàn toàn đến từ khác biệt giữa các FT seed: r0 có ngưỡng khoảng 0.524 và dừng ở bước 526–620, còn r1, r2 có ngưỡng khoảng 0.59–0.60 và dừng ở bước 468–549. Mức p = 0.008 giả định 15 điểm độc lập. Thực tế 15 điểm gom thành 3 nhóm theo FT seed, nên p này quá lạc quan.
  - Trong một FT seed, ngưỡng gần như không đổi (lệch dưới 0.5%), vậy mà bước dừng vẫn trải rộng tới 94 bước ở r0 (526–620) và 81 bước ở r1. Khác biệt này đi theo mode của bước 09 (QD24): ở r0, seed 0–2 (có cụm crease) dừng ở 593–620, còn seed 3–4 dừng ở 526–531; ở r1, seed 0–2 dừng ở 534–549, còn seed 3–4 ở 468–486.
  - [SUY LUẬN] Vậy bước dừng của lượt S phụ thuộc hai thứ. Giữa các FT seed, nó phụ thuộc ngưỡng, mà ngưỡng lại do ngân sách tối ưu quyết định là chính (06 §3.2). Trong một FT seed, nó phụ thuộc dữ liệu mà lượt S học, tức phân hoạch ở 09 và ảnh được bộ lọc giữ lại. Một cải tiến ở 09 (E09.06, E09.07) vì thế cũng đổi luôn ngân sách bước của lượt S, như đã nêu khi phản biện kế hoạch. 13 sẽ kiểm bằng quỹ đạo det.

## Summary (EN)

1. **Step 07 only names the rule; the rule acts inside passes F and S.** `rerun` reproduces the `stop` key (`stop_default`). The rule compares the mean det of the old classes (from `global_params`) with the mean det of the 5 new slots. When the old det is below 1 it stops once the new det falls below old det / sf, with sf = 2 in pass F and √(5/10) = 0.707 in pass S. It can fire only after step 333 and rolls back one update (§1.1, §2.1).
2. **`--scaling-factor` is dead (Q07.02).** `self.scaling_factor` and `self.early_stop_ratio` are assigned in `mngmm.py` and never read; `main.py` only prints and forwards the flag. The factors of the rule are computed from the class counts (§2.1).
3. **The code stops pass S earlier than Eq. 11.** With the old det 0.3707 of r0 s4 and new slots starting at det 1, the thresholds are 0.524 for pass S (1.414 × old), 0.374 for Eq. 11 (R < 0.01, 1.010 × old) and 0.185 for pass F (0.5 × old). Pass S therefore stops while the new covariances are still 41% "larger" than the old ones, not at the equality the paper argues for (D14, §2.2).
4. **Across 15 snapshots pass F never stops and pass S always does.** Pass S stops at steps 468–620. The S threshold and the stop step correlate at Spearman −0.66, mostly between FT seeds (thresholds 0.524 for r0 vs 0.59–0.60 for r1, r2). Within an FT seed the threshold varies by under 0.5% while the stop step spans up to 94 steps and follows the step-09 mode (§3.1). [inference] Changes at step 09 also change the pass-S step budget; checked with det trajectories in 13.